In [9]:
import duckdb, glob
con = duckdb.connect()
files = sorted(glob.glob("../data/raw/lycos-ids2017/*.csv"))
print("найдено файлов:", len(files))   # должно быть 5
for f in files:
    print(f)
    print(con.execute(f"SELECT label, COUNT(*) n FROM read_csv_auto('{f}') GROUP BY label ORDER BY n DESC").df().to_string(index=False))
    print()

найдено файлов: 5
../data/raw/lycos-ids2017/Friday-WorkingHours.pcap_lycos.csv
   label      n
  benign 286438
portscan 160106
    ddos  95683
     bot    735

../data/raw/lycos-ids2017/Monday-WorkingHours.pcap_lycos.csv
 label      n
benign 367181

../data/raw/lycos-ids2017/Thursday-WorkingHours.pcap_lycos.csv
                  label      n
                 benign 110448
   webattack_bruteforce   1360
          webattack_xss    661
webattack_sql_injection     12

../data/raw/lycos-ids2017/Tuesday-WorkingHours.pcap_lycos.csv
      label      n
     benign 312675
ftp_patator   4003
ssh_patator   2959

../data/raw/lycos-ids2017/Wednesday-WorkingHours.pcap_lycos.csv
           label      n
          benign 318933
        dos_hulk 158988
   dos_goldeneye   6765
   dos_slowloris   5674
dos_slowhttptest   4866
      heartbleed     11



In [10]:
# ── Обработка размеченного LycoS-IDS2017 → processed/2017.parquet ─────────────
# Вход: 5 размеченных CSV (labelling.py от Rosay) в data/raw/lycos-ids2017/.
# Выход: 2017.parquet на тех же 76 признаках, что и 2018, в том же порядке,
#        с сохранённым sentinel -1, с label (стиль 2018) и label_binary.
import sys; sys.path.insert(0, "..")
import glob
from src import config, data

con = data.get_duckdb_connection()

# Эталон: порядок и состав 76 признаков из processed-2018.
feat_2018 = data.get_feature_cols(con)   # 76 признаков в каноническом порядке

# Размеченные CSV.
csv_files = sorted(glob.glob(f"{config.RAW_2017_DIR}/*.pcap_lycos.csv"))
assert len(csv_files) == 5, f"ожидали 5 CSV, нашли {len(csv_files)}"
print("Файлы 2017:")
for f in csv_files:
    print(" ", f)

def _q(name):
    return '"' + name.replace('"', '""') + '"'

# CASE-выражение маппинга меток 2017 → стиль 2018.
label_case = "CASE label\n" + "\n".join(
    f"        WHEN '{k}' THEN '{v}'" for k, v in config.LABEL_MAP_2017.items()
) + "\n        ELSE label\n    END"

# Список 76 признаков в порядке 2018 (читаются из CSV по имени — порядок гарантирован).
feat_select = ", ".join(_q(c) for c in feat_2018)

# read_csv_auto по всем 5 файлам разом (DuckDB принимает список путей).
files_sql = "[" + ", ".join(f"'{f}'" for f in csv_files) + "]"

con.execute(f"""
    COPY (
        SELECT
            {feat_select},
            ({label_case}) AS label,
            CASE WHEN label = 'benign' THEN 0 ELSE 1 END AS label_binary
        FROM read_csv_auto({files_sql}, union_by_name=true)
    )
    TO '{config.CROSS_2017_PATH}' (FORMAT PARQUET, COMPRESSION SNAPPY)
""")

n = data.count_rows(con, config.CROSS_2017_PATH)
print(f"\n2017.parquet создан: {n:,} строк → {config.CROSS_2017_PATH}")

Файлы 2017:
  /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/raw/lycos-ids2017/Friday-WorkingHours.pcap_lycos.csv
  /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/raw/lycos-ids2017/Monday-WorkingHours.pcap_lycos.csv
  /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/raw/lycos-ids2017/Thursday-WorkingHours.pcap_lycos.csv
  /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/raw/lycos-ids2017/Tuesday-WorkingHours.pcap_lycos.csv
  /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/raw/lycos-ids2017/Wednesday-WorkingHours.pcap_lycos.csv

2017.parquet создан: 1,837,498 строк → /home/kirill/Рабочий стол/Диплом/traffic-classification-vkr_v2.0/data/processed/2017.parquet


In [11]:
# ── Верификация 2017.parquet ──────────────────────────────────────────────────
# 1. Схема: ровно 76 признаков + label + label_binary, как в 2018.
cols_2017 = con.execute(
    f"SELECT * FROM read_parquet('{config.CROSS_2017_PATH}') LIMIT 0"
).df().columns.tolist()
feat_check = [c for c in cols_2017 if c not in ("label", "label_binary")]

assert feat_check == feat_2018, "порядок/состав признаков 2017 != 2018"
print(f"Схема ✓  признаков: {len(feat_check)} (совпадают с 2018 по составу и порядку)")

# 2. Sentinel -1 сохранён (не затёрт).
neg = con.execute(f"""
    SELECT COUNT(*) FROM read_parquet('{config.CROSS_2017_PATH}')
    WHERE "fwd_tcp_init_win_bytes" < 0 OR "bwd_tcp_init_win_bytes" < 0
""").fetchone()[0]
print(f"Sentinel -1 сохранён: {neg:,} строк {'✓' if neg > 0 else '✗ (проверь!)'}")

# 3. Распределение меток + бинарный баланс.
dist = con.execute(f"""
    SELECT label, label_binary, COUNT(*) AS n,
           ROUND(100.0*COUNT(*)/SUM(COUNT(*)) OVER (), 2) AS pct
    FROM read_parquet('{config.CROSS_2017_PATH}')
    GROUP BY label, label_binary ORDER BY n DESC
""").df()
print("\nРаспределение меток 2017:")
print(dist.to_string(index=False))

mal = con.execute(f"SELECT AVG(label_binary) FROM read_parquet('{config.CROSS_2017_PATH}')").fetchone()[0]
print(f"\nMalicious доля: {mal:.4f}  (2018 train был ~0.27)")

# 4. NaN/Inf санити (как в ноутбуке 1).
bad = con.execute(f"""
    SELECT
        COUNT(*) FILTER (WHERE isinf(bytes_per_s) OR isnan(bytes_per_s)) AS bad_bps,
        COUNT(*) FILTER (WHERE isinf(pkt_per_s)   OR isnan(pkt_per_s))   AS bad_pps
    FROM read_parquet('{config.CROSS_2017_PATH}')
""").fetchone()
print(f"NaN/Inf в bytes_per_s/pkt_per_s: {bad[0]}, {bad[1]} {'✓' if bad==(0,0) else '◄ проверь'}")

Схема ✓  признаков: 76 (совпадают с 2018 по составу и порядку)
Sentinel -1 сохранён: 898,080 строк ✓

Распределение меток 2017:
                     label  label_binary       n   pct
                    Benign             0 1395675 75.96
                  PortScan             1  160106  8.71
                  DoS Hulk             1  158988  8.65
                      DDoS             1   95683  5.21
             DoS GoldenEye             1    6765  0.37
             DoS Slowloris             1    5674  0.31
          DoS Slowhttptest             1    4866  0.26
               FTP-Patator             1    4003  0.22
               SSH-Patator             1    2959  0.16
  Web Attack - Brute Force             1    1360  0.07
                       Bot             1     735  0.04
          Web Attack - XSS             1     661  0.04
Web Attack - Sql Injection             1      12  0.00
                Heartbleed             1      11  0.00

Malicious доля: 0.2404  (2018 train был ~0.27)

In [12]:
# ── Cross-dataset: модели 2018 → тест на LycoS-IDS2017 ────────────────────────
# Модели и порог НЕ трогаем. Препроцессор тот же (preprocessor.pkl от 2018).
# Порог заморожен на val-2018 (из baseline). evaluate сам прогонит 2017.
import sys; sys.path.insert(0, "..")
import json, joblib
import numpy as np
import torch
from src import config, data, preprocess, evaluate, mlp

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con, config.CROSS_2017_PATH)  # 76, проверка инвариантов
pre = preprocess.load_preprocessor()
CROSS = config.CROSS_2017_PATH

# Замороженные пороги из baseline-val (берём из сохранённых *_val.json).
def val_threshold(model_key):
    with open(config.BASELINE_DIR / f"{model_key}_val.json") as f:
        return json.load(f)["threshold"]

results_cross = {}

In [13]:
# ── LogReg ────────────────────────────────────────────────────────────────────
logreg = joblib.load(config.MODELS_DIR / "logreg.pkl")
def predict_logreg(X): return logreg.predict_proba(pre.transform(X))[:, 1]

results_cross["LogReg"] = evaluate.evaluate(
    "logreg_cross", predict_logreg, CROSS, feature_cols,
    threshold=val_threshold("logreg"), split_name="cross2017",
)
del logreg


  logreg_cross  (cross2017)
строк 1,837,498 за 3.96s  |  порог 0.6175 (frozen_from_val)
F1_macro 0.7821  MCC 0.569  PR-AUC 0.582679  ROC-AUC 0.879418  FPR@95 0.284116
TN=1,199,787  FP=195,888  FN=114,288  TP=327,535  (FPR=0.1404, FNR=0.2587)

per-class recall:
  Benign                           n=1,395,675  recall=85.96%
  PortScan                         n= 160,106  recall=50.65%
  DoS Hulk                         n= 158,988  recall=99.82%
  DDoS                             n=  95,683  recall=74.70%
  DoS GoldenEye                    n=   6,765  recall=96.87%
  DoS Slowloris                    n=   5,674  recall=5.94% ◄
  DoS Slowhttptest                 n=   4,866  recall=8.98% ◄
  FTP-Patator                      n=   4,003  recall=99.80%
  SSH-Patator                      n=   2,959  recall=99.93%
  Web Attack - Brute Force         n=   1,360  recall=94.71%
  Bot                              n=     735  recall=5.31% ◄
  Web Attack - XSS                 n=     661  recall=97.28%
  

In [14]:
# ── RandomForest (tree-view, без препроцессора) ───────────────────────────────
rf = joblib.load(config.MODELS_DIR / "rf.pkl")
def predict_rf(X): return rf.predict_proba(X)[:, 1]

results_cross["RandomForest"] = evaluate.evaluate(
    "rf_cross", predict_rf, CROSS, feature_cols,
    threshold=val_threshold("rf"), split_name="cross2017",
)
del rf

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.7s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.7s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.4s finished



  rf_cross  (cross2017)
строк 1,837,498 за 4.94s  |  порог 0.39 (frozen_from_val)
F1_macro 0.5899  MCC 0.3691  PR-AUC 0.805756  ROC-AUC 0.898711  FPR@95 1.0
TN=1,395,197  FP=478  FN=365,129  TP=76,694  (FPR=0.0003, FNR=0.8264)

per-class recall:
  Benign                           n=1,395,675  recall=99.97%
  PortScan                         n= 160,106  recall=44.90% ◄
  DoS Hulk                         n= 158,988  recall=0.01% ◄
  DDoS                             n=  95,683  recall=0.23% ◄
  DoS GoldenEye                    n=   6,765  recall=1.42% ◄
  DoS Slowloris                    n=   5,674  recall=61.86%
  DoS Slowhttptest                 n=   4,866  recall=19.24% ◄
  FTP-Patator                      n=   4,003  recall=0.00% ◄
  SSH-Patator                      n=   2,959  recall=0.00% ◄
  Web Attack - Brute Force         n=   1,360  recall=0.22% ◄
  Bot                              n=     735  recall=0.00% ◄
  Web Attack - XSS                 n=     661  recall=2.57% ◄
  Web At

In [15]:
# ── XGBoost (tree-view) ───────────────────────────────────────────────────────
xgb_model = joblib.load(config.MODELS_DIR / "xgb.pkl")
def predict_xgb(X): return xgb_model.predict_proba(X)[:, 1]

results_cross["XGBoost"] = evaluate.evaluate(
    "xgb_cross", predict_xgb, CROSS, feature_cols,
    threshold=val_threshold("xgb"), split_name="cross2017",
)
del xgb_model


  xgb_cross  (cross2017)
строк 1,837,498 за 3.04s  |  порог 0.7613 (frozen_from_val)
F1_macro 0.4366  MCC 0.0202  PR-AUC 0.702404  ROC-AUC 0.885526  FPR@95 0.267392
TN=1,391,958  FP=3,717  FN=439,448  TP=2,375  (FPR=0.0027, FNR=0.9946)

per-class recall:
  Benign                           n=1,395,675  recall=99.73%
  PortScan                         n= 160,106  recall=0.26% ◄
  DoS Hulk                         n= 158,988  recall=0.01% ◄
  DDoS                             n=  95,683  recall=0.29% ◄
  DoS GoldenEye                    n=   6,765  recall=0.00% ◄
  DoS Slowloris                    n=   5,674  recall=28.08% ◄
  DoS Slowhttptest                 n=   4,866  recall=1.29% ◄
  FTP-Patator                      n=   4,003  recall=0.00% ◄
  SSH-Patator                      n=   2,959  recall=0.00% ◄
  Web Attack - Brute Force         n=   1,360  recall=0.00% ◄
  Bot                              n=     735  recall=0.00% ◄
  Web Attack - XSS                 n=     661  recall=0.00% ◄

In [16]:
# ── MLP (linear-view) ─────────────────────────────────────────────────────────
mlp_model = mlp.MLP(pre.n_features_out_)
mlp_model.load_state_dict(torch.load(config.MODELS_DIR / "mlp.pt"))
mlp_model.eval()
predict_mlp = mlp.make_predict_fn(mlp_model, pre, device="cpu")

results_cross["MLP"] = evaluate.evaluate(
    "mlp_cross", predict_mlp, CROSS, feature_cols,
    threshold=val_threshold("mlp"), split_name="cross2017",
)


  mlp_cross  (cross2017)
строк 1,837,498 за 5.21s  |  порог 0.9175 (frozen_from_val)
F1_macro 0.5814  MCC 0.2186  PR-AUC 0.483841  ROC-AUC 0.832694  FPR@95 0.34278
TN=1,308,358  FP=87,317  FN=346,592  TP=95,231  (FPR=0.0626, FNR=0.7845)

per-class recall:
  Benign                           n=1,395,675  recall=93.74%
  PortScan                         n= 160,106  recall=50.47%
  DoS Hulk                         n= 158,988  recall=0.01% ◄
  DDoS                             n=  95,683  recall=0.00% ◄
  DoS GoldenEye                    n=   6,765  recall=42.66% ◄
  DoS Slowloris                    n=   5,674  recall=56.59%
  DoS Slowhttptest                 n=   4,866  recall=24.52% ◄
  FTP-Patator                      n=   4,003  recall=99.73%
  SSH-Patator                      n=   2,959  recall=99.90%
  Web Attack - Brute Force         n=   1,360  recall=11.10% ◄
  Bot                              n=     735  recall=0.00% ◄
  Web Attack - XSS                 n=     661  recall=3.18% ◄


In [17]:
# ── Within (2018 test) vs Cross (2017) — главная таблица работы ───────────────
print("="*72)
print("  WITHIN (2018 test)  vs  CROSS (2017)")
print("="*72)
print(f"\n  {'Модель':<14}{'within MCC':>12}{'cross MCC':>11}{'within F1':>11}{'cross F1':>10}")
print("  " + "-"*56)
for key, name in [("logreg","LogReg"),("rf","RandomForest"),("xgb","XGBoost"),("mlp","MLP")]:
    with open(config.BASELINE_DIR / f"{key}_test.json") as f:
        w = json.load(f)
    c = results_cross[name]
    print(f"  {name:<14}{w['MCC']:>12.4f}{c['MCC']:>11.4f}{w['F1_macro']:>11.4f}{c['F1_macro']:>10.4f}")

  WITHIN (2018 test)  vs  CROSS (2017)

  Модель          within MCC  cross MCC  within F1  cross F1
  --------------------------------------------------------
  LogReg              0.9383     0.5690     0.9684    0.7821
  RandomForest        0.9970     0.3691     0.9985    0.5899
  XGBoost             0.9970     0.0202     0.9985    0.4366
  MLP                 0.9961     0.2186     0.9981    0.5814


In [18]:
# ── Cross с ОПТИМАЛЬНЫМ порогом на 2017 (для сопоставимости с Cantone) ─────────
# threshold=None → evaluate подбирает порог на самом 2017 (как у Cantone).
# Это ОПТИМИСТИЧНАЯ оценка (подгонка порога под тест), в отличие от честной
# операционной оценки с замороженным порогом 2018. Приводим обе.
import joblib, torch
from src import config, data, preprocess, evaluate, mlp

con = data.get_duckdb_connection()
feature_cols = data.get_feature_cols(con, config.CROSS_2017_PATH)
pre = preprocess.load_preprocessor()
CROSS = config.CROSS_2017_PATH

results_opt = {}

# LogReg
logreg = joblib.load(config.MODELS_DIR / "logreg.pkl")
results_opt["LogReg"] = evaluate.evaluate(
    "logreg_cross_opt", lambda X: logreg.predict_proba(pre.transform(X))[:, 1],
    CROSS, feature_cols, threshold=None, split_name="cross2017_opt", verbose=False,
)
del logreg

# RandomForest
rf = joblib.load(config.MODELS_DIR / "rf.pkl")
results_opt["RandomForest"] = evaluate.evaluate(
    "rf_cross_opt", lambda X: rf.predict_proba(X)[:, 1],
    CROSS, feature_cols, threshold=None, split_name="cross2017_opt", verbose=False,
)
del rf

# XGBoost
xgb_model = joblib.load(config.MODELS_DIR / "xgb.pkl")
results_opt["XGBoost"] = evaluate.evaluate(
    "xgb_cross_opt", lambda X: xgb_model.predict_proba(X)[:, 1],
    CROSS, feature_cols, threshold=None, split_name="cross2017_opt", verbose=False,
)
del xgb_model

# MLP
mlp_model = mlp.MLP(pre.n_features_out_)
mlp_model.load_state_dict(torch.load(config.MODELS_DIR / "mlp.pt"))
mlp_model.eval()
results_opt["MLP"] = evaluate.evaluate(
    "mlp_cross_opt", mlp.make_predict_fn(mlp_model, pre, device="cpu"),
    CROSS, feature_cols, threshold=None, split_name="cross2017_opt", verbose=False,
)

print("Готово — оптимальные пороги на 2017 подобраны.")

[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.2s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.6s finished
[Parallel(n_jobs=16)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=16)]: Done  18 tasks      | elapsed:    0.1s
[Parallel(n_jobs=16)]: Done 100 out of 100 | elapsed:    0.4s finished


Готово — оптимальные пороги на 2017 подобраны.


In [19]:
# ── Финальная таблица: within vs cross(frozen) vs cross(opt) vs Cantone ───────
import json

# Числа Cantone (LycoS18 → LycoS17) из его Table — для сравнения.
# LDA сопоставляем с нашим LogReg (обе простые линейные).
CANTONE = {
    "LogReg":       {"mcc": 0.6041, "note": "LDA у Cantone"},
    "RandomForest": {"mcc": 0.3833, "note": ""},
    "XGBoost":      {"mcc": 0.4029, "note": ""},
    "MLP":          {"mcc": None,   "note": "нет у Cantone"},
}

print("="*92)
print("  ИТОГ ЭТАПА 1:  within (2018) → cross (2017)")
print("="*92)
print(f"\n  {'Модель':<13}{'within':>9}{'cross':>9}{'cross':>9}{'Cantone':>9}   "
      f"{'cross AUROC':>12}")
print(f"  {'':<13}{'MCC':>9}{'frozen':>9}{'opt':>9}{'MCC':>9}   {'(порог-незав.)':>12}")
print("  " + "-"*72)

for key, name in [("logreg","LogReg"),("rf","RandomForest"),("xgb","XGBoost"),("mlp","MLP")]:
    with open(config.BASELINE_DIR / f"{key}_test.json") as f:
        within = json.load(f)["MCC"]
    with open(config.BASELINE_DIR / f"{key}_cross_cross2017.json") as f:
        frozen = json.load(f)
    opt = results_opt[name]
    cant = CANTONE[name]["mcc"]
    cant_s = f"{cant:>8.4f}" if cant is not None else f"{'—':>8}"
    print(f"  {name:<13}{within:>9.4f}{frozen['MCC']:>9.4f}{opt['MCC']:>9.4f}{cant_s}   "
          f"{frozen['ROC_AUC']:>12.4f}")

print("\n  frozen = честная операционная оценка (порог заморожен на val-2018)")
print("  opt    = оптимистичная оценка (порог подобран на 2017, как у Cantone)")

  ИТОГ ЭТАПА 1:  within (2018) → cross (2017)

  Модель          within    cross    cross  Cantone    cross AUROC
                     MCC   frozen      opt      MCC   (порог-незав.)
  ------------------------------------------------------------------------
  LogReg          0.9383   0.5690   0.5760  0.6041         0.8794
  RandomForest    0.9970   0.3691   0.7330  0.3833         0.8987
  XGBoost         0.9970   0.0202   0.6141  0.4029         0.8855
  MLP             0.9961   0.2186   0.5464       —         0.8327

  frozen = честная операционная оценка (порог заморожен на val-2018)
  opt    = оптимистичная оценка (порог подобран на 2017, как у Cantone)


In [21]:
# ── Сводка cross-dataset (2018 → 2017): всё в один JSON ───────────────────────
import sys; sys.path.insert(0, "..")
import json
from src import config

MODELS = [("logreg","LogReg"), ("rf","RandomForest"), ("xgb","XGBoost"), ("mlp","MLP")]

CANTONE_MCC = {"LogReg": 0.6041, "RandomForest": 0.3833, "XGBoost": 0.4029, "MLP": None}

def load(name):
    with open(config.BASELINE_DIR / name) as f:
        return json.load(f)

# Собираем по каждой модели: within(test) + cross frozen + cross opt.
models_summary = {}
for key, name in MODELS:
    within = load(f"{key}_test.json")
    frozen = load(f"{key}_cross_cross2017.json")
    opt = load(f"{key}_cross_opt_cross2017_opt.json")
    models_summary[name] = {
        "within_2018":  {"MCC": within["MCC"], "F1_macro": within["F1_macro"],
                          "ROC_AUC": within["ROC_AUC"], "threshold": within["threshold"]},
        "cross_frozen": {"MCC": frozen["MCC"], "F1_macro": frozen["F1_macro"],
                         "ROC_AUC": frozen["ROC_AUC"], "threshold": frozen["threshold"],
                         "FNR": frozen["binary"]["FNR"], "FPR": frozen["binary"]["FPR"]},
        "cross_opt":    {"MCC": opt["MCC"], "F1_macro": opt["F1_macro"],
                         "threshold": opt["threshold"]},
        "cantone_ref_MCC": CANTONE_MCC[name],
        "per_class_cross": frozen["per_class_recall"],   # frozen = операционный сценарий
    }

# ── Per-class анализ переносимости (по frozen, операционный сценарий) ──────────
# Для каждой атаки — recall у 4 моделей на 2017. Видно, что переносится, что нет.
any_pc = models_summary["LogReg"]["per_class_cross"]
classes = sorted(any_pc.items(), key=lambda x: -x[1]["n"])

per_class_table = {}
for cls, d in classes:
    row = {"n": d["n"]}
    for _, name in MODELS:
        row[name] = models_summary[name]["per_class_cross"].get(cls, {}).get("recall")
    per_class_table[cls] = row

summary = {
    "experiment": "cross-dataset generalization",
    "train": "LycoS-Unicas-IDS2018", "test": "LycoS-IDS2017",
    "test_rows": load("logreg_cross_cross2017.json")["n_rows"],
    "threshold_policy": {
        "frozen": "порог заморожен на val-2018 (честная операционная оценка)",
        "opt":    "порог подобран на 2017 (оптимистичная, сопоставима с Cantone)",
    },
    "models": models_summary,
    "per_class_transferability": per_class_table,
}

out = config.CROSS_DIR / "cross_summary.json"
with open(out, "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

# ── Печать: главная таблица ───────────────────────────────────────────────────
print("="*84)
print("  CROSS-DATASET: LycoS18 → LycoS17")
print("="*84)
print(f"\n  {'Модель':<14}{'within':>9}{'frozen':>9}{'opt':>9}{'Cantone':>9}{'AUROC':>9}{'FNR_frozen':>12}")
print("  " + "-"*70)
for _, name in MODELS:
    m = models_summary[name]
    cant = m["cantone_ref_MCC"]
    cant_s = f"{cant:>9.4f}" if cant is not None else f"{'—':>9}"
    print(f"  {name:<14}{m['within_2018']['MCC']:>9.4f}{m['cross_frozen']['MCC']:>9.4f}"
          f"{m['cross_opt']['MCC']:>9.4f}{cant_s}{m['cross_frozen']['ROC_AUC']:>9.4f}"
          f"{m['cross_frozen']['FNR']:>12.4f}")

# ── Печать: per-class переносимость (frozen) ──────────────────────────────────
print("\n" + "="*84)
print("  PER-CLASS RECALL на 2017 (frozen) — что переносится, что нет")
print("="*84)
print(f"\n  {'класс':<26}{'n':>9}" + "".join(f"{name:>13}" for _, name in MODELS))
print("  " + "-"*(35 + 13*len(MODELS)))
for cls, row in per_class_table.items():
    line = f"  {cls:<26}{row['n']:>9,}"
    for _, name in MODELS:
        r = row[name]
        line += f"{r:>12.1%} " if r is not None else f"{'—':>13}"
    print(line)

print(f"\n→ cross_summary.json сохранён: {out}")

  CROSS-DATASET: LycoS18 → LycoS17

  Модель           within   frozen      opt  Cantone    AUROC  FNR_frozen
  ----------------------------------------------------------------------
  LogReg           0.9383   0.5690   0.5760   0.6041   0.8794      0.2587
  RandomForest     0.9970   0.3691   0.7330   0.3833   0.8987      0.8264
  XGBoost          0.9970   0.0202   0.6141   0.4029   0.8855      0.9946
  MLP              0.9961   0.2186   0.5464        —   0.8327      0.7845

  PER-CLASS RECALL на 2017 (frozen) — что переносится, что нет

  класс                             n       LogReg RandomForest      XGBoost          MLP
  ---------------------------------------------------------------------------------------
  Benign                    1,395,675       86.0%       100.0%        99.7%        93.7% 
  PortScan                    160,106       50.6%        44.9%         0.3%        50.5% 
  DoS Hulk                    158,988       99.8%         0.0%         0.0%         0.0% 
  DDoS